# 🎭 Sentiment Analysis — NLP Pipeline

**Dataset:** IMDB Movie Reviews (20,000 reviews)  
**Models:** Naive Bayes with TF-IDF features  
**NLP:** spaCy preprocessing pipeline

---

## 0. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from wordcloud import WordCloud

# NLP
import spacy

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
import joblib

# Plot style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 12

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models',  exist_ok=True)

print('✅ All libraries loaded.')

## 1. Load Data

In [ ]:
DATA_PATH   = '../data/IMDB Dataset.csv'
SAMPLE_SIZE = 20_000

df = pd.read_csv(DATA_PATH)
print(f'Full dataset shape: {df.shape}')
df.head(3)

In [ ]:
# Sub-sample for speed
df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f'Working dataset: {df.shape}')
print('\nMissing values:\n', df.isnull().sum())
print('\nLabel distribution:\n', df['sentiment'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
# ── Sentiment distribution ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

counts = df['sentiment'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Sentiment Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')

# Review length distribution
df['review_len'] = df['review'].str.split().str.len()
for label, color in [('positive', 'steelblue'), ('negative', 'tomato')]:
    sub = df[df['sentiment'] == label]['review_len']
    axes[1].hist(sub.clip(0, 500), bins=40, alpha=0.6, label=label, color=color, edgecolor='white')
axes[1].set_title('Review Length Distribution')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/eda_overview.png')
plt.show()

## 3. Preprocessing — spaCy NLP Pipeline

In [ ]:
# Load spaCy (disable heavy components we don't need)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
print('spaCy model loaded ✅')

def raw_clean(text: str) -> str:
    """Remove HTML, URLs, and special characters."""
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)          # HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text) # URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)   # non-alpha
    text = re.sub(r'\s+', ' ', text).strip()    # whitespace
    return text

def spacy_clean(texts: list, batch_size: int = 500) -> list:
    """Tokenise, lemmatise, and remove stopwords/punctuation via spaCy."""
    cleaned = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        tokens = [
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and not token.is_space
            and len(token.lemma_) > 1
        ]
        cleaned.append(' '.join(tokens))
    return cleaned

# Apply cleaning
print('Step 1/2 — Basic cleaning…')
df['raw_clean'] = df['review'].apply(raw_clean)

print('Step 2/2 — spaCy lemmatisation (may take ~30s)…')
df['clean_text'] = spacy_clean(df['raw_clean'].tolist())

# Encode label
df['label'] = (df['sentiment'] == 'positive').astype(int)

# Drop empty rows
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)

print(f'\n✅ Preprocessing done. {len(df):,} rows remain.')
print(df[['review', 'clean_text', 'sentiment']].head(2))

## 4. Word Clouds

In [ ]:
def make_wordcloud(text_series, title, colormap='Blues'):
    text = ' '.join(text_series.dropna())
    wc = WordCloud(
        width=900, height=400, background_color='white',
        colormap=colormap, max_words=150, collocations=False
    ).generate(text)
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(title, fontsize=15, fontweight='bold')
    plt.tight_layout()
    fname = title.lower().replace(' ', '_') + '.png'
    plt.savefig(f'../outputs/{fname}')
    plt.show()

make_wordcloud(df[df['label']==1]['clean_text'], 'Positive Review WordCloud', 'Greens')
make_wordcloud(df[df['label']==0]['clean_text'], 'Negative Review WordCloud', 'Reds')

## 5. Feature Engineering — TF-IDF

In [ ]:
X = df['clean_text']
y = df['label']

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Class balance (train): {y_train.mean():.1%} positive')

# TF-IDF (unigrams + bigrams)
print('\nFitting TF-IDF vectorizer…')
vectorizer = TfidfVectorizer(
    max_features = 20_000,
    ngram_range  = (1, 2),
    sublinear_tf = True,
    min_df       = 2,
    max_df       = 0.95,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Train matrix: {X_train_tfidf.shape}')
print(f'Test  matrix: {X_test_tfidf.shape}')

## 6. Naive Bayes Classifier

In [ ]:
model = MultinomialNB(alpha=0.1)
model.fit(X_train_tfidf, y_train)
print('MultinomialNB trained ✅')

# Predict
y_pred = model.predict(X_test_tfidf)

# Metrics
acc  = accuracy_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred, average='weighted')
prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)

print(f'\nAccuracy  : {acc:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print('\nDetailed Report:')
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

## 7. Confusion Matrix & Feature Importance

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=ax)
ax.set_title('Confusion Matrix — Naive Bayes', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png')
plt.show()

In [ ]:
# ── Top informative features ──────────────────────────────────────────
feature_names  = np.array(vectorizer.get_feature_names_out())
log_prob_diff  = model.feature_log_prob_[1] - model.feature_log_prob_[0]

n = 20
top_pos_idx = np.argsort(log_prob_diff)[-n:]
top_neg_idx = np.argsort(log_prob_diff)[:n]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, idx, color, title in [
    (axes[0], top_pos_idx, 'steelblue', f'Top {n} Positive Features'),
    (axes[1], top_neg_idx, 'tomato',    f'Top {n} Negative Features'),
]:
    words  = feature_names[idx]
    scores = log_prob_diff[idx]
    ax.barh(words, np.abs(scores), color=color, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Log-Prob Difference (abs)')
    ax.invert_yaxis()

plt.suptitle('Most Informative TF-IDF Features (Naive Bayes)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/top_features.png', bbox_inches='tight')
plt.show()

## 8. Save Model

In [ ]:
joblib.dump(model,      '../models/nb_model.joblib')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.joblib')
print('✅ Model saved → models/nb_model.joblib')
print('✅ Vectorizer saved → models/tfidf_vectorizer.joblib')

## 9. Inference — Predict New Reviews

In [ ]:
def predict_review(text: str) -> None:
    raw   = raw_clean(text)
    clean = spacy_clean([raw])[0]
    X     = vectorizer.transform([clean])
    pred  = model.predict(X)[0]
    prob  = model.predict_proba(X)[0].max()
    label = 'Positive ✅' if pred == 1 else 'Negative ❌'
    print(f'Review    : {text}')
    print(f'Sentiment : {label}  (confidence: {prob:.1%})\n')

# Test examples
predict_review('This movie was absolutely fantastic! One of the best films I have seen.')
predict_review('Terrible acting, boring plot. Complete waste of time.')
predict_review('It was okay. Nothing special, but not terrible either.')

## 10. Summary

In [ ]:
print('=' * 55)
print('  SENTIMENT ANALYSIS — SUMMARY')
print('=' * 55)
print(f'Dataset  : IMDB Movie Reviews ({len(df):,} reviews)')
print(f'NLP      : spaCy (en_core_web_sm) — lemmatisation + stopword removal')
print(f'Features : TF-IDF  unigrams + bigrams  (20,000 vocab)')
print(f'Split    : 80% train / 20% test  (stratified)')
print(f'Model    : Naive Bayes (MultinomialNB, alpha=0.1)')
print()
print(f'Accuracy  : {acc:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print()
print('Outputs saved in outputs/ directory.')
print('Models  saved in models/  directory.')